In [ ]:
import mujoco
import numpy as np

model = mujoco.MjModel.from_xml_path("../robot.xml")
data = mujoco.MjData(model)

# set a known configuration
data.qpos[:] = [0.0, 0.0, 0.0, 0.0]

site_name = "ee"
site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, site_name)

# compute nominal kinematics
mujoco.mj_forward(model, data)
x0 = data.site_xpos[site_id].copy()

# compute Jacobian at this configuration
jacp = np.zeros((3, model.nv))
jacr = np.zeros((3, model.nv))
mujoco.mj_jacSite(model, data, jacp, jacr, site_id)

print("jacp:")
print(jacp)
print("jacp norm:", np.linalg.norm(jacp))

# small perturbation for Jacobian check
eps = 1e-6
delta_qpos = np.random.randn(model.nv) * eps

q0 = data.qpos.copy()
data.qpos[:] = q0 + delta_qpos
mujoco.mj_forward(model, data)

x1 = data.site_xpos[site_id].copy()

delta_x_actual = x1 - x0
delta_x_pred = jacp @ delta_qpos
error = delta_x_actual - delta_x_pred

print("\nActual delta x:")
print(delta_x_actual)

print("\nPredicted delta x from Jacobian:")
print(delta_x_pred)

print("\nDifference (actual - predicted):")
print(error)

print("\nDifference norm:")
print(np.linalg.norm(error))

# restore original state
data.qpos[:] = q0
mujoco.mj_forward(model, data)

jacp:
[[-5.63000000e-02  9.72555370e-18 -5.39568390e-17 -4.16226298e-18]
 [ 8.68467000e-05 -9.30000000e-02 -6.40000000e-02 -6.40000000e-02]
 [ 0.00000000e+00  2.19000000e-02  1.21500000e-01  0.00000000e+00]]
jacp norm: 0.18775664446924173

Actual delta x:
[ 5.24541426e-08 -2.40556085e-08  3.30472889e-08]

Predicted delta x from Jacobian:
[ 5.24541650e-08 -2.40555884e-08  3.30472984e-08]

Difference (actual - predicted):
[-2.23994258e-14 -2.00990480e-14 -9.57090431e-15]

Difference norm:
3.158018708017702e-14
